# Day 3 Lab: Advanced SQL, Schema & Nested Data in Databricks

## Objectives
- Work with nested JSON data
- Apply explicit schemas
- Use Spark SQL + DataFrames
- Handle NULLs and complex types


In [0]:
data = [
{'order_id':1,'customer':{'id':101,'name':'Alice'},'items':[{'product':'A','price':100},{'product':'B','price':150}],'city':'Bangalore'},
{'order_id':2,'customer':{'id':102,'name':'Bob'},'items':[{'product':'C','price':200}],'city':'Chennai'},
{'order_id':3,'customer':{'id':103,'name':'Charlie'},'items':None,'city':'Hyderabad'}
]

df = spark.createDataFrame(data)
display(df)


In [0]:
from pyspark.sql.types import *

schema = StructType([
 StructField('order_id', IntegerType()),
 StructField('customer', StructType([
   StructField('id', IntegerType()),
   StructField('name', StringType())
 ])),
 StructField('items', ArrayType(StructType([
   StructField('product', StringType()),
   StructField('price', IntegerType())
 ]))),
 StructField('city', StringType())
])

df_schema = spark.createDataFrame(
    data,
    schema=schema
)
display(df_schema)


In [0]:
from pyspark.sql.functions import explode

flat_df = df_schema.withColumn('item', explode('items'))

result_df = flat_df.select(
 'order_id',
 'customer.id',
 'customer.name',
 'city',
 'item.product',
 'item.price'
)

display(result_df)


In [0]:
clean_df = df_schema.na.fill({'city':'Unknown'})
display(clean_df)


In [0]:
result_df.createOrReplaceTempView('orders')


In [0]:
%sql
SELECT city, SUM(price) as revenue
FROM orders
GROUP BY city


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

window_spec = Window.partitionBy('city').orderBy(col('price').desc())

ranked_df = result_df.withColumn('rank', row_number().over(window_spec))
display(ranked_df)


In [0]:
# mention the appropriate catalog, schema and volume names
# Update with proper catalog, schema and volume names
# Either use existing catalog or create a catalog, existing schema or create a new schema, existing volume or create a new volume.
 
result_df.write.format('delta').mode('overwrite').save('/Volumes/<catalog_name>/<schema_name>/<volume_name>/orders_delta')

# result_df.write.format('delta').mode('overwrite').save('/Volumes/dbcuser01/ingesting_data/myfiles/orders_delta')